In [2]:
from llama_index.core import Settings, StorageContext, load_index_from_storage
from llama_index.embeddings.ollama import OllamaEmbedding

OLLAMA_EMBED_MODEL = "embeddinggemma"
embed_model = OllamaEmbedding(
  model_name=OLLAMA_EMBED_MODEL, 
  embed_batch_size=1024
)

Settings.embed_model = embed_model

chunk_size=100_000
index_dir_name=f"vector_index_coco_{chunk_size}_nodes_ollama_{OLLAMA_EMBED_MODEL}"

In [3]:
# load storage context from disk
storage_context_full_dir = f"../storage/{index_dir_name}"
print(storage_context_full_dir)
storage_context = StorageContext.from_defaults(persist_dir=storage_context_full_dir)

../storage/vector_index_coco_100000_nodes_ollama_embeddinggemma


In [4]:
# build index from storage context
# these steps are not needed if you have the index object already
index = load_index_from_storage(storage_context)


In [12]:
# add trace
from openinference.instrumentation.llama_index import LlamaIndexInstrumentor
from phoenix.otel import register

tracer_provider = register()
LlamaIndexInstrumentor().instrument(tracer_provider=tracer_provider)

🔭 OpenTelemetry Tracing Details 🔭
|  Phoenix Project: default
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: localhost:4317
|  Transport: gRPC
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  ⚠️ WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



In [13]:
from llama_index.llms.ollama import Ollama

OLLAMA_LLM_MODEL = "qwen3:1.7b"
llm_model = Ollama(
  model=OLLAMA_LLM_MODEL,
  request_timeout=300,
)

response = llm_model.complete("Explain the theory of relativity in simple terms." )
print(response)

**********
Trace: completion
    |_CBEventType.LLM -> 4.639153 seconds
    |_CBEventType.LLM -> 4.635663 seconds
**********
The **Theory of Relativity** is a set of two major ideas developed by **Albert Einstein** that changed how we understand space, time, and gravity. Let me break it down in simple terms:

---

### **1. Special Relativity (about constant speeds):**  
- **What it says:** When objects move at speeds close to the speed of light (about 300,000 km/s), time and space "stretch" or "contract" depending on how fast you're moving.  
- **Examples:**  
  - A spaceship traveling at 90% the speed of light would experience time **slower** than someone on Earth.  
  - If you shine a light in a spaceship, the light would appear to move **faster** from your perspective, but in Earth's frame, it still moves at the speed of light.  
- **Key Idea:** Time and space are **not absolute**—they change depending on your **speed** and **location**.

---

### **2. General Relativity (about gravi

In [14]:
query_engine = index.as_query_engine(llm=llm_model)

async def search_images(query: str) -> str:
    """Search images based on descriptions of the objects in the image."""
    response = query_engine.query(query)
    return str(response)

def addition(a: int, b: int) -> int:
    """Used to add 2 numbers."""
    return a + b

In [15]:
from llama_index.core.agent import AgentWorkflow
from llama_index.core.callbacks import CallbackManager, LlamaDebugHandler

debug_handler = LlamaDebugHandler(print_trace_on_end=True)
Settings.callback_manager = CallbackManager([debug_handler])

agent = AgentWorkflow.from_tools_or_functions(
  [addition, search_images,], llm=llm_model,
  verbose=True,
  system_prompt="You are a helpful assistant that can answer questions about images and do simple math calculations."
)

In [18]:

response = await agent.run(
    "What is 25 plus 30? Also, describe an image that contains a cat sitting on a windowsill with sunlight streaming in."
)
print(response)




**********
Trace: chat
    |_CBEventType.LLM -> 0.0 seconds
**********
**********
Trace: query
    |_CBEventType.QUERY -> 9.054902 seconds
      |_CBEventType.SYNTHESIZE -> 6.526145 seconds
        |_CBEventType.TEMPLATING -> 1.2e-05 seconds
        |_CBEventType.LLM -> 6.519223 seconds
**********
**********
Trace: chat
    |_CBEventType.LLM -> 0.0 seconds
**********
The result of $25 + 30$ is $\boxed{55}$. 

For the image description, the query "a cat sitting on a windowsill with sunlight streaming in" was used, even though the context does not explicitly mention sunlight. The answer reflects the details provided in the query.


In [19]:

response = await agent.run(
    "What is 10 plus 30? Also, find an image with a dog riding a bicycle."
)
print(response)




**********
Trace: chat
    |_CBEventType.LLM -> 0.0 seconds
**********
**********
Trace: query
    |_CBEventType.QUERY -> 3.259611 seconds
      |_CBEventType.SYNTHESIZE -> 1.367716 seconds
        |_CBEventType.TEMPLATING -> 1.1e-05 seconds
        |_CBEventType.LLM -> 1.350153 seconds
**********
**********
Trace: chat
    |_CBEventType.LLM -> 0.0 seconds
**********
The result of $10 + 30$ is **40**. 

An image of a dog riding a bicycle has been found in the context.


In [17]:
events = debug_handler.get_event_pairs()
print("Num events:", len(events))
for i, event in enumerate(events):
    print(event)

Num events: 2
[CBEvent(event_type=<CBEventType.CHUNKING: 'chunking'>, payload={<EventPayload.CHUNKS: 'chunks'>: ['row_id: 32165\ncaption_slot: caption1\n\na cat that is sitting in a window sill\n\nrow_id: 31817\ncaption_slot: caption1\n\nA cat sitting on top of an open window sill.']}, time='01/26/2026, 15:05:09.213505', id_='a719eb02-eb46-4b22-bdcf-cee74ba40727'), CBEvent(event_type=<CBEventType.CHUNKING: 'chunking'>, payload={<EventPayload.CHUNKS: 'chunks'>: ['row_id: 32165\ncaption_slot: caption1\n\na cat that is sitting in a window sill\n\nrow_id: 31817\ncaption_slot: caption1\n\nA cat sitting on top of an open window sill.']}, time='01/26/2026, 15:05:09.213569', id_='a719eb02-eb46-4b22-bdcf-cee74ba40727')]
[CBEvent(event_type=<CBEventType.CHUNKING: 'chunking'>, payload={<EventPayload.CHUNKS: 'chunks'>: ['row_id: 32165\ncaption_slot: caption1\n\na cat that is sitting in a window sill\n\nrow_id: 31817\ncaption_slot: caption1\n\nA cat sitting on top of an open window sill.']}, time='